In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# ⚛️ H7 Parking Violations: Metriplectic QML & Gemma 4
**Track**: Safety & Trust
**Concept**: Zero-Shot Civic Governance Classification via Metriplectic Information Theory


In [ ]:
import os
import json
import logging
import random
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel, SafetySetting, Part
except ImportError:
    vertexai = None

try:
    from transformers import AutoTokenizer, pipeline
    from optimum.onnxruntime import ORTModelForCausalLM
    LOCAL_ONNX_SUPPORT = True
except ImportError:
    LOCAL_ONNX_SUPPORT = False

logger = logging.getLogger(__name__)

class GemmaMetriplexOracle:
    """
    Oráculo de Información usando Gemma 4.
    Traduce lenguaje natural (infracciones) a las densidades iniciales (rho y v)
    requeridas para el Clasificador QML Metripléctico.
    """
    def __init__(self):
        self.project_id = os.getenv("VERTEX_PROJECT_ID")
        self.region = os.getenv("VERTEX_REGION", "us-central1")
        self.model_name = os.getenv("GEMMA_BASE_MODEL", "gemma-4-26b-a4b-it")
        self.use_mock = False
        self.use_offline_onnx = False
        self.local_onnx_pipeline = None

        if not self.project_id or vertexai is None:
            logger.warning("Credenciales de Vertex AI no encontradas o vertexai no instalado. Activando fallback Offline (ONNX).")
            self._init_offline_onnx()
        else:
            try:
                vertexai.init(project=self.project_id, location=self.region)
                publisher_model_name = f"publishers/google/models/gemma4@{self.model_name.lower()}"
                self.model = GenerativeModel(publisher_model_name)
            except Exception as e:
                logger.error(f"Fallo al inicializar VertexAI: {e}. Activando fallback Offline (ONNX).")
                self._init_offline_onnx()

    def _init_offline_onnx(self):
        """Inicializa el modelo Gemma E2B/E4B de forma local y offline usando ONNX Runtime."""
        if LOCAL_ONNX_SUPPORT:
            try:
                logger.info("Inicializando ONNX Runtime para Gemma...")
                # Se puede usar gemma-2b-it u otro como default para Edge
                edge_model_id = os.getenv("GEMMA_EDGE_MODEL", "google/gemma-2b-it") 
                
                # Cargamos tokenizer y el ORTModel (Optimizado para ONNX)
                tokenizer = AutoTokenizer.from_pretrained(edge_model_id)
                # NOTA: download_kwargs o uso de modelos locales exportados
                onnx_model = ORTModelForCausalLM.from_pretrained(edge_model_id, export=True) 
                
                self.local_onnx_pipeline = pipeline(
                    "text-generation",
                    model=onnx_model,
                    tokenizer=tokenizer,
                    max_new_tokens=100
                )
                self.use_offline_onnx = True
                logger.warning("ONNX Edge Model cargado correctamente en memoria local.")
            except Exception as e:
                logger.error(f"Fallo carga profunda de ONNX (¿pesos muy grandes o sin conexión inicial?): {e}. Usando mock temporal.")
                self.use_mock = True
        else:
            logger.error("ONNX/Transformers no instalados o no configurados. Usando mock.")
            self.use_mock = True

    def get_initial_phase_state(self, context: str, domain_prompt: str = "Analiza el siguiente contexto:") -> dict:
        """
        Consulta a Gemma 4 para obtener las variables físicas:
        rho: Densidad de la infracción (magnitud/gravedad percibida, 0 a 1)
        v: Velocidad/Intencionalidad (flujo de comportamiento, -1 a 1)
        """
        if self.use_mock:
            # Hash del contexto para mantener consistencia en la simulación
            h = sum(ord(c) for c in context)
            random.seed(h)
            rho = round(random.uniform(0.1, 1.0), 4)
            v = round(random.uniform(-1.0, 1.0), 4)
            return {"rho": rho, "v": v}

        prompt = f"""
        Eres un Oráculo Físico Metripléctico.
        {domain_prompt}
        "{context}"
        
        Evalúa y extrae exclusivamente dos valores en formato JSON estricto:
        1. "rho": Densidad de impacto o peso real en el sistema (de 0.1 a 1.0).
        2. "v": Intencionalidad o dirección del flujo asociado (-1.0 a 1.0, donde negativo es perjudicial/destructivo y positivo es ordenado/constructivo).
        
        Ejemplo: {{"rho": 0.8, "v": -0.5}}
        Solo responde con el JSON.
        """

        try:
            if self.use_offline_onnx and self.local_onnx_pipeline:
                # Inferencia puramente OFF-LINE, por CPU / Edge via ONNX
                response = self.local_onnx_pipeline(prompt)
                content = response[0]["generated_text"].replace(prompt, "").strip()
            else:
                # Inferencia CLOUD (Vertex AI Gemma 4)
                response = self.model.generate_content(
                    prompt,
                    generation_config={"temperature": 0.2, "max_output_tokens": 100}
                )
                content = response.text.strip()

            # Limpieza básica por si el modelo envuelve en markdown
            if content.startswith("```json"):
                content = content[7:-3]
            elif content.startswith("```"):
                content = content[3:-3]
                
            data = json.loads(content)
            return {"rho": float(data.get("rho", 0.5)), "v": float(data.get("v", 0.0))}
        except Exception as e:
            logger.error(f"Error consultando al Oráculo (ONNX/Cloud falló): {e}")
            return {"rho": 0.5, "v": 0.0}

In [ ]:
import os
import json
import logging
import random
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ====================================================================
# 1. ORÁCULO DE INFORMACIÓN (GEMMA 4)
# ====================================================================
try:
    import vertexai
    from vertexai.generative_models import GenerativeModel, SafetySetting, Part
except ImportError:
    vertexai = None

try:
    from transformers import AutoTokenizer, pipeline
    from optimum.onnxruntime import ORTModelForCausalLM
    LOCAL_ONNX_SUPPORT = True
except ImportError:
    LOCAL_ONNX_SUPPORT = False

logger = logging.getLogger(__name__)

class GemmaMetriplexOracle:
    """
    Oráculo de Información usando Gemma 4.
    Traduce lenguaje natural (infracciones) a las densidades iniciales (rho y v)
    requeridas para el Clasificador QML Metripléctico.
    """
    def __init__(self):
        self.project_id = os.getenv("VERTEX_PROJECT_ID")
        self.region = os.getenv("VERTEX_REGION", "us-central1")
        self.model_name = os.getenv("GEMMA_BASE_MODEL", "gemma-4-26b-a4b-it")
        self.use_mock = False
        self.use_offline_onnx = False
        self.local_onnx_pipeline = None

        if not self.project_id or vertexai is None:
            logger.warning("Credenciales de Vertex AI no encontradas o vertexai no instalado. Activando fallback Offline (ONNX).")
            self._init_offline_onnx()
        else:
            try:
                vertexai.init(project=self.project_id, location=self.region)
                publisher_model_name = f"publishers/google/models/gemma4@{self.model_name.lower()}"
                self.model = GenerativeModel(publisher_model_name)
            except Exception as e:
                logger.error(f"Fallo al inicializar VertexAI: {e}. Activando fallback Offline (ONNX).")
                self._init_offline_onnx()

    def _init_offline_onnx(self):
        """Inicializa el modelo Gemma E2B/E4B de forma local y offline usando ONNX Runtime."""
        if LOCAL_ONNX_SUPPORT:
            try:
                logger.info("Inicializando ONNX Runtime para Gemma...")
                edge_model_id = os.getenv("GEMMA_EDGE_MODEL", "google/gemma-2b-it") 
                tokenizer = AutoTokenizer.from_pretrained(edge_model_id)
                onnx_model = ORTModelForCausalLM.from_pretrained(edge_model_id, export=True) 
                
                self.local_onnx_pipeline = pipeline(
                    "text-generation",
                    model=onnx_model,
                    tokenizer=tokenizer,
                    max_new_tokens=100
                )
                self.use_offline_onnx = True
                logger.warning("ONNX Edge Model cargado correctamente en memoria local.")
            except Exception as e:
                logger.error(f"Fallo carga profunda de ONNX. Usando mock temporal. Error: {e}")
                self.use_mock = True
        else:
            logger.error("ONNX/Transformers no instalados o no configurados. Usando mock.")
            self.use_mock = True

    def get_initial_phase_state(self, context: str, domain_prompt: str = "Analiza el siguiente contexto:") -> dict:
        if self.use_mock:
            h = sum(ord(c) for c in context)
            random.seed(h)
            rho = round(random.uniform(0.1, 1.0), 4)
            v = round(random.uniform(-1.0, 1.0), 4)
            return {"rho": rho, "v": v}

        prompt = f"""
Eres un Oráculo de Información Metripléctico para Gobernanza Cívica.
Analiza esta infracción municipal y extrae dos parámetros físicos:

Contexto: "{context}"

1. **Densidad (ρ)**: Magnitud real/impacto del incidente [0.1 a 1.0]
   - 0.1-0.3: Trivial/administrativo (ej "Meter expirado")
   - 0.3-0.7: Moderado (ej "Estacionamiento doble")
   - 0.7-1.0: Crítico (ej "Bloqueando acceso de emergencia")

2. **Velocidad/Intencionalidad (v)**: Dirección moral [-1.0 a 1.0]
   - -1.0: Claramente malintencionado/peligroso
   - -0.5 a 0: Negligencia o infracción sin excusa
   - 0 a +0.5: Casos grises, hay mitigantes
   - +1.0: Circunstancias atenuantes documentadas

Responde SOLO JSON: {{"rho": 0.X, "v": -0.X}}
"""

        try:
            if self.use_offline_onnx and self.local_onnx_pipeline:
                response = self.local_onnx_pipeline(prompt)
                content = response[0]["generated_text"].replace(prompt, "").strip()
            else:
                response = self.model.generate_content(
                    prompt,
                    generation_config={"temperature": 0.2, "max_output_tokens": 100}
                )
                content = response.text.strip()

            if content.startswith("```json"):
                content = content[7:-3]
            elif content.startswith("```"):
                content = content[3:-3]
                
            data = json.loads(content)
            return {"rho": float(data.get("rho", 0.5)), "v": float(data.get("v", 0.0))}
        except Exception as e:
            logger.error(f"Error consultando al Oráculo (ONNX/Cloud falló): {e}")
            return {"rho": 0.5, "v": 0.0}

# ====================================================================
# 2. CLASIFICADOR QML METRIPLÉCTICO (H7 PROTOCOL)
# ====================================================================
class H7TernaryClassifier:
    """
    H7 Metriplectic QML Classifier.
    Implementa el Mandato Metripléctico donde la dinámica está regida por:
    - Lagrangiano explícito: Componente Simpléctica (H) y Métrica (S).
    - Topología O_n con la Razón Áurea.
    
    Predice la clase {-1, 0, 1} según el atractor final del estado.
    """
    def __init__(self, epsilon: float = 0.25):
        self.epsilon = epsilon
        self.phi = (1 + np.sqrt(5)) / 2
        self.PI = np.pi
        
        self.history_symp = []
        self.history_metr = []
        self.history_psi = []

    def golden_operator(self, n: float) -> float:
        """Operador O_n que modula el fondo estructurado del espacio de fase."""
        return np.cos(self.PI * n) * np.cos(self.PI * self.phi * n)

    def compute_lagrangian(self, psi: float, rho: float, v: float):
        """
        Calcula d_symp (fuerzas conservativas) y d_metr (fuerzas disipativas).
        """
        d_symp = rho * v * np.sin(self.PI * psi)
        attractor_force = - (psi - np.round(psi)) 
        d_metr = (1.0 - rho) * attractor_force + 0.1 * v * self.golden_operator(psi)
        
        return d_symp, d_metr
        
    def fit_predict(self, rho: float, v: float, steps: int = 50, dt: float = 0.1) -> int:
        """
        Ejecuta la dinámica para que la partícula de información caiga
        en uno de los tres atractores de estado (-1, 0, 1).
        """
        self.history_symp = []
        self.history_metr = []
        self.history_psi = []
        
        psi_current = v * self.golden_operator(rho * 10)
        
        for _ in range(steps):
            d_symp, d_metr = self.compute_lagrangian(psi_current, rho, v)
            
            self.history_symp.append(float(d_symp))
            self.history_metr.append(float(d_metr))
            self.history_psi.append(float(psi_current))
            
            delta_psi = (d_symp + d_metr) * dt
            psi_current += delta_psi
            psi_current = np.clip(psi_current, -1.5, 1.5)
            
        final_psi = psi_current
        self.history_psi.append(float(final_psi))
        
        if final_psi > self.epsilon:
            return 1
        elif final_psi < -self.epsilon:
            return -1
        else:
            return 0
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import time

# --- Celda de Interacción Metripléctica (Kaggle Widgets) ---

# Pre-computamos/simulamos caché para no quemar la API de Vertex al mover los sliders
oracle_cache = {}
def get_cached_metrics(context):
    if context not in oracle_cache:
        oracle_cache[context] = oracle.get_initial_phase_state(context)
    return oracle_cache[context]

# 1. Menú de tickets
ticket_choices = [
    (f"#{t['id'] if 'id' in t else i} | {t['description'][:60]}...", t) 
    for i, t in df.iterrows()
] if isinstance(df, pd.DataFrame) else [
    (f"#{t.get('id', i)} | {t['description'][:60]}...", t) for i, t in enumerate(sample_data)
]

w_ticket = widgets.Dropdown(options=ticket_choices, description='Ticket:')

# 2. Deslizadores Metriplécticos
w_steps   = widgets.IntSlider(value=50, min=10, max=150, step=10, description='Pasos:')
w_dt      = widgets.FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='dt:')
w_epsilon = widgets.FloatSlider(value=0.25, min=0.05, max=0.8, step=0.05, description='Epsilon ε:')

output_plot = widgets.Output()

def update_widget_plot(change):
    with output_plot:
        clear_output(wait=True)
        ticket = w_ticket.value
        
        # 1. Oracle Extraction
        metrics = get_cached_metrics(ticket["context"])
        rho, v = metrics["rho"], metrics["v"]
        
        # 2. Dynamics Evolution
        clf = H7TernaryClassifier(epsilon=w_epsilon.value)
        cls = clf.fit_predict(rho, v, steps=w_steps.value, dt=w_dt.value)
        
        # 3. Rendereado Mágico Interactivo (Plotly interactivo en celda)
        fig = go.Figure()
        
        # Energía
        fig.add_trace(go.Scatter(y=clf.history_symp, name="L_symp (Conservativo)", 
                                 line=dict(color="#1D9E75", width=2)))
        # Entropía
        fig.add_trace(go.Scatter(y=clf.history_metr, name="L_metr (Disipativo)", 
                                 line=dict(color="#D85A30", width=2)))
        # Trayectoria de Estado
        fig.add_trace(go.Scatter(y=clf.history_psi, name="ψ(t) (Fase)", 
                                 line=dict(color="#7F77DD", width=4)))
        
        # Atractores
        fig.add_hline(y=1, line_dash="dash", line_color="green", annotation_text="Atractor +1 (Constructivo)")
        fig.add_hline(y=-1, line_dash="dash", line_color="red", annotation_text="Atractor -1 (Destructivo)")
        fig.add_hline(y=0, line_dash="dash", line_color="orange", annotation_text="Atractor 0 (Equilibrio)")
        
        fig.update_layout(
            title=f"<b>Resolución de Gobernanza:</b> ρ={rho:.3f}, v={v:.3f} ➔ <b>Clase Resultante: {cls:+d}</b>",
            template="plotly_dark",
            height=450,
            xaxis_title=f"Unidades de Tiempo (dt={w_dt.value})",
            yaxis_title="Topología del Espacio Analítico H7",
            hovermode="x unified",
            margin=dict(l=20, r=20, t=50, b=20)
        )
        
        # Muestra en Output
        fig.show()

# Enlazar listeners para que el gráfico se actualice "en vivo" si algo se desliza
w_ticket.observe(update_widget_plot, names='value')
w_steps.observe(update_widget_plot, names='value')
w_dt.observe(update_widget_plot, names='value')
w_epsilon.observe(update_widget_plot, names='value')

# Poner una estructura bonita visualmente
controls = widgets.VBox([
    widgets.HTML("<h3>🧬 Inspector Zero-Shot H7 de Interactividad</h3>"),
    w_ticket,
    widgets.HBox([w_steps, w_dt, w_epsilon])
])

# Desplegar en el notebook!
display(controls, output_plot)
update_widget_plot(None) # Ejecución inicial
